<a href="https://colab.research.google.com/github/brasilemnumeros/dados/blob/main/indicadores/fiscal/Resultado-Primario.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Resultado primário do setor público — % do PIB

Este notebook é o motor de coleta e publicação da série `resultado-primario-pib`.

- **Fonte:** Banco Central do Brasil — Estatísticas Fiscais, via API de Dados Abertos do BCB
  (SGS), série **5793** — “NFSP sem desvalorização cambial (% PIB) — fluxo acumulado em 12
  meses — resultado primário — total — setor público consolidado”.
- **Endpoint:** `https://api.bcb.gov.br/dados/serie/bcdata.sgs.5793/dados?formato=json`
- **Definição:** receitas menos despesas do setor público consolidado — União, INSS, estados,
  municípios e estatais (exceto Petrobras e Eletrobras) — **sem** os juros da dívida,
  acumuladas nos 12 meses encerrados no mês e divididas pelo PIB corrente dos mesmos 12
  meses. `percentual` (`%`, 2 casas), `eixo: percentual`, `medida: acumulado-12m`,
  granularidade `mensal`.
- **Fórmula:** nenhuma de nível. O valor é o da fonte com o **sinal invertido**: o SGS publica
  a necessidade de financiamento (NFSP), em que positivo é déficit; a série publica positivo
  para superávit, que é a convenção de “resultado primário”.
- **Período publicado:** de `2002-11-01` até o último mês fechado disponível.
- **Arquivo gerado:** `indicadores/fiscal/resultado-primario-pib.json`, com `versao: 1` e
  `id: resultado-primario-pib`.

## O que o número é — e o que não é

- **É um fluxo acumulado em 12 meses**, não um estoque: mede o esforço fiscal do período, não
  a dívida. A dívida líquida e a bruta são séries próprias.
- **Exclui os juros da dívida.** O resultado nominal é o primário mais os juros nominais; um
  primário positivo pode conviver com nominal muito negativo.
- **Positivo é superávit, negativo é déficit.** O sinal é o da convenção de resultado
  primário, não o da NFSP do SGS.
- **É o setor público consolidado**, não o governo central: inclui estados, municípios,
  estatais e Banco Central. O resultado do governo central é outro recorte.
- **O denominador é o PIB corrente de 12 meses do BCB.** Quando o IBGE revisa as Contas
  Nacionais, a razão inteira pode mudar; é revisão da fonte, não erro de fórmula.
- **2020 é atípico** por causa do gasto emergencial da pandemia; **2021 e 2022** têm superávit
  inflado por receitas extraordinárias e pela inflação alta. A leitura de tendência atravessa
  esses choques, não os ignora.
- **Meses recentes são preliminares** e podem ser revisados na divulgação seguinte.

## Coleta das séries oficiais

A leitura é feita desde o início da série, sem atualização incremental e sem ler o JSON já
publicado para decidir o período. O SGS devolve `data` em `DD/MM/AAAA` e `valor` como texto; o
valor é convertido para `float` sem arredondar.

Além da série publicada, o notebook busca duas séries irmãs do mesmo produtor para a
conferência de identidade: **5727** (resultado nominal, % do PIB, 12 meses) e **5760** (juros
nominais, % do PIB, 12 meses). Elas não são publicadas.

In [1]:
from datetime import date, datetime
from pathlib import Path
import json
import math
import os
import re
import time

import pandas as pd
import requests

RAIZ = Path(os.environ.get("DADOS_RAIZ", Path.cwd())).resolve()

if os.environ.get("DADOS_RAIZ"):
    PASTA = RAIZ / "indicadores" / "fiscal"
    if not PASTA.is_dir():
        raise SystemExit(f"família não encontrada em {PASTA}")
else:
    PASTA = RAIZ

DESTINO = Path(os.environ.get("FISCAL_SAIDA", PASTA)).resolve()
ARQUIVO = DESTINO / "resultado-primario-pib.json"


def relativo(caminho):
    """Caminho exibido no notebook: relativo à raiz, nunca o absoluto da máquina."""
    try:
        return caminho.relative_to(RAIZ)
    except ValueError:
        return caminho.name

In [2]:
SERIE = 5793
SERIE_NOMINAL = 5727
SERIE_JUROS = 5760
BASE = "https://api.bcb.gov.br/dados/serie/bcdata.sgs.{serie}/dados"
URL_FONTE = BASE.format(serie=SERIE) + "?formato=json"
HOJE = date.today()


def baixar(serie, tentativas=4, timeout=60):
    """GET no SGS com espera crescente; 200 com HTML também cai no except."""
    url = BASE.format(serie=serie)
    for tentativa in range(tentativas):
        try:
            resposta = requests.get(url, params={"formato": "json"}, timeout=timeout)
            resposta.raise_for_status()
            bruto = resposta.json()
            if not isinstance(bruto, list) or not bruto:
                raise ValueError("resposta vazia ou fora do formato de lista")
            return pd.DataFrame(
                {
                    "data": pd.to_datetime(
                        [datetime.strptime(item["data"], "%d/%m/%Y") for item in bruto]
                    ),
                    "valor": [float(item["valor"]) for item in bruto],
                }
            ).sort_values("data", ignore_index=True)
        except (requests.RequestException, ValueError):
            if tentativa == tentativas - 1:
                raise
            time.sleep(2 ** tentativa)


df = baixar(SERIE)
nominal = baixar(SERIE_NOMINAL)
juros = baixar(SERIE_JUROS)

print(f"Série SGS {SERIE}: {len(df)} leituras")
print(f"Período bruto: {df['data'].min():%Y-%m-%d} a {df['data'].max():%Y-%m-%d}")
print(f"Arquivo de saída: {relativo(ARQUIVO)}")
print("\nPrimeiras 5 leituras (sinal da fonte, positivo = déficit):")
print(df.head().to_string(index=False))
print("\nÚltimas 5 leituras:")
print(df.tail().to_string(index=False))

Série SGS 5793: 287 leituras
Período bruto: 2002-11-01 a 2026-09-01
Arquivo de saída: indicadores/fiscal/resultado-primario-pib.json

Primeiras 5 leituras (sinal da fonte, positivo = déficit):
      data  valor
2002-11-01  -3.31
2002-12-01  -3.19
2003-01-01  -3.25
2003-02-01  -3.36
2003-03-01  -3.33

Últimas 5 leituras:
      data  valor
2026-05-01   1.14
2026-06-01   1.19
2026-07-01   0.67
2026-08-01   0.62
2026-09-01   0.52


## Filtragem dos meses fechados e inversão do sinal

O mês corrente é descartado porque ainda está aberto. A fonte devolve as datas no primeiro dia
de cada mês; essa convenção é normalizada e preservada em C15. A inversão de sinal é feita uma
única vez, aqui: `publicado = −fonte`.

A conferência de identidade usa as três séries do mesmo mês:
`NFSP primária + juros nominais = NFSP nominal`. A tolerância é de 0,02 p.p., o suficiente para
o arredondamento de três séries publicadas com 2 casas.

In [3]:
primeiro_dia_mes_corrente = pd.Timestamp(HOJE.replace(day=1))
fechados = df[df["data"] < primeiro_dia_mes_corrente].copy()
fechados["mes"] = fechados["data"].dt.to_period("M")
fechados["publicado"] = -fechados["valor"]

if fechados.empty:
    raise ValueError("a fonte não retornou nenhum mês fechado")

if fechados["mes"].duplicated().any():
    raise ValueError("a fonte retornou meses repetidos")

# Identidade contábil entre séries irmãs do mesmo produtor, mês a mês.
nominal_por_mes = dict(zip(nominal["data"], nominal["valor"]))
juros_por_mes = dict(zip(juros["data"], juros["valor"]))
for data, primario in zip(fechados["data"], fechados["valor"]):
    if data not in nominal_por_mes or data not in juros_por_mes:
        raise ValueError(f"sem resultado nominal ou juros para {data:%Y-%m}")
    desvio = primario + juros_por_mes[data] - nominal_por_mes[data]
    if abs(desvio) > 0.02:
        raise ValueError(f"{data:%Y-%m}: primário + juros - nominal = {desvio:.2f}")

print(f"Mês corrente excluído: {HOJE:%Y-%m}")
print(f"Meses fechados publicados: {len(fechados)}")
print(
    f"Primeiro: {fechados['data'].iloc[0]:%Y-%m-%d} "
    f"(fonte {fechados['valor'].iloc[0]:.2f}%, publicado {fechados['publicado'].iloc[0]:.2f}%)"
)
print(
    f"Último:   {fechados['data'].iloc[-1]:%Y-%m-%d} "
    f"(fonte {fechados['valor'].iloc[-1]:.2f}%, publicado {fechados['publicado'].iloc[-1]:.2f}%)"
)
print("Identidade primário + juros = nominal conferida em todos os meses (tolerância 0,02 p.p.).")

Mês corrente excluído: 2026-11
Meses fechados publicados: 287
Primeiro: 2002-11-01 (fonte -3.31%, publicado 3.31%)
Último:   2026-09-01 (fonte 0.52%, publicado -0.52%)
Identidade primário + juros = nominal conferida em todos os meses (tolerância 0,02 p.p.).


## Escrita do JSON canônico

Os pontos são gravados como `{"data": "YYYY-MM-DD", "valor": ...}`, em ordem ascendente, com a
data no primeiro dia do mês e sem campos extras. O valor publicado é o da fonte com o sinal
invertido, sem arredondamento; `unidade.casas` é só exibição.

In [4]:
pontos = [
    {"data": f"{data.year:04d}-{data.month:02d}-01", "valor": float(valor)}
    for data, valor in zip(fechados["data"], fechados["publicado"])
]

serie = {
    "versao": 1,
    "id": "resultado-primario-pib",
    "unidade": {"tipo": "percentual", "simbolo": "%", "casas": 2},
    "eixo": "percentual",
    "medida": "acumulado-12m",
    "granularidade": "mensal",
    "fonte": (
        "Banco Central do Brasil — Estatísticas Fiscais (SGS 5793); "
        "sinal invertido pelo site (positivo = superávit)"
    ),
    "urlFonte": URL_FONTE,
    "dados": pontos,
}

DESTINO.mkdir(parents=True, exist_ok=True)
with ARQUIVO.open("w", encoding="utf-8") as arquivo:
    json.dump(serie, arquivo, ensure_ascii=False, indent=2)

print(f"{len(pontos)} pontos escritos em {relativo(ARQUIVO)}")
print(f"Primeiro ponto: {pontos[0]}")
print(f"Último ponto:   {pontos[-1]}")

287 pontos escritos em indicadores/fiscal/resultado-primario-pib.json
Primeiro ponto: {'data': '2002-11-01', 'valor': 3.31}
Último ponto:   {'data': '2026-09-01', 'valor': -0.52}


## Validação inline

A célula final falha se o contrato, os metadados, a convenção C15, a unicidade dos meses, a
ordem, a finitude dos valores, o primeiro ponto, os canários históricos ou a igualdade com a
fonte forem violados. Os canários cobrem o pico do déficit (2020-12), o superávit de 2022
(2022-12), o fechamento de 2025 (2025-12) e setembro de 2026 (2026-09).

O notebook também confere a identidade `primário + juros = nominal` contra as SGS 5727 e 5760
em todos os meses, e que o valor publicado é exatamente o negativo do valor da fonte.

In [5]:
CHAVES = {
    "versao", "id", "unidade", "eixo", "medida", "granularidade", "fonte", "urlFonte", "dados",
}
FONTE = (
    "Banco Central do Brasil — Estatísticas Fiscais (SGS 5793); "
    "sinal invertido pelo site (positivo = superávit)"
)
assert set(serie) == CHAVES, sorted(set(serie) ^ CHAVES)
assert serie["versao"] == 1, serie["versao"]
assert serie["id"] == "resultado-primario-pib", serie["id"]
assert serie["unidade"] == {"tipo": "percentual", "simbolo": "%", "casas": 2}, serie["unidade"]
assert serie["eixo"] == "percentual", serie["eixo"]
assert serie["medida"] == "acumulado-12m", serie["medida"]
assert serie["granularidade"] == "mensal", serie["granularidade"]
assert serie["fonte"] == FONTE, serie["fonte"]
assert serie["urlFonte"] == "https://api.bcb.gov.br/dados/serie/bcdata.sgs.5793/dados?formato=json", serie["urlFonte"]

datas = [ponto["data"] for ponto in serie["dados"]]
meses = [data[:7] for data in datas]

assert datas, "série vazia"
assert datas == sorted(datas), "datas fora de ordem"
assert all(a < b for a, b in zip(datas, datas[1:])), "datas não estritamente ascendentes"
assert len(meses) == len(set(meses)), "mês repetido"
assert datas[0] == "2002-11-01", datas[0]

# Canários independentes: pontos históricos fixos, fora do `fechados` que gerou a série.
por_data = {ponto["data"]: ponto["valor"] for ponto in serie["dados"]}
assert por_data["2002-11-01"] == 3.31, por_data["2002-11-01"]
assert por_data["2020-12-01"] == -9.24, por_data["2020-12-01"]
assert por_data["2022-12-01"] == 1.25, por_data["2022-12-01"]
assert por_data["2025-12-01"] == -0.43, por_data["2025-12-01"]
assert por_data.get("2026-09-01") == -0.52, por_data.get("2026-09-01")

assert datas[-1][:7] < f"{HOJE:%Y-%m}", datas[-1]
assert len(datas) == len(fechados), (len(datas), len(fechados))

for ponto in serie["dados"]:
    assert set(ponto) == {"data", "valor"}, sorted(ponto)
    assert re.fullmatch(r"[0-9]{4}-[0-9]{2}-[0-9]{2}", ponto["data"]), ponto
    assert datetime.strptime(ponto["data"], "%Y-%m-%d").day == 1, ponto
    assert not isinstance(ponto["valor"], bool), ponto
    assert isinstance(ponto["valor"], (int, float)), ponto
    assert math.isfinite(ponto["valor"]), ponto

# O publicado é o negativo da fonte, ponto a ponto.
esperado_da_fonte = [
    {"data": f"{data.year:04d}-{data.month:02d}-01", "valor": float(-valor)}
    for data, valor in zip(fechados["data"], fechados["valor"])
]
assert serie["dados"] == esperado_da_fonte, "valor publicado difere da fonte"

# Identidade contra as séries irmãs, no objeto que vai ao disco.
for ponto in serie["dados"]:
    data = pd.Timestamp(ponto["data"])
    primario_nfsp = -ponto["valor"]
    desvio = primario_nfsp + juros_por_mes[data] - nominal_por_mes[data]
    assert abs(desvio) <= 0.02, (ponto["data"], desvio)

with ARQUIVO.open(encoding="utf-8") as arquivo:
    salvo = json.load(arquivo)
assert salvo == serie, "arquivo gravado difere do objeto validado"

print(
    f"ok: {len(datas)} meses únicos, {datas[0]} a {datas[-1]}, "
    f"valores iguais ao negativo da SGS 5793 e identidade primário + juros = nominal conferida"
)

ok: 287 meses únicos, 2002-11-01 a 2026-09-01, valores iguais ao negativo da SGS 5793 e identidade primário + juros = nominal conferida
